# Trail Activity Prediction Visualizer

Interactive notebook for inspecting one completed activity at segment level.

The prediction uses the paper-extension speed equation with the activity's observed segment HRR values. Acute TRIMP fatigue is accumulated from predicted segment times, not actual segment times, so the segment prediction does not leak the true segment duration into the fatigue state. The visualized acute state is a cumulative performance reserve: at fixed HRR and equal terrain, later segments get slower as the reserve decreases.

Use the config cell to set `ACTIVITY_ID`, or use the optional selector at the bottom when `ipywidgets` is available.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from services import trail_performance_model as tpm  # noqa: E402

DATA_DIR = ROOT / "data"
TIMESERIES_DIR = DATA_DIR / "timeseries"
PAPER_ASSET_DIR = ROOT / "docs" / "science" / "paper_assets"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## Configuration

`ACTIVITY_ID` defaults to the Lyon Urban Trail By Night activity if present. Set it to any activity with a file in `data/timeseries/`.

In [ ]:
ACTIVITY_ID = "16325125849"
SEGMENT_KM = 1.0

# Stage 3 defaults are read from docs/science/paper_assets/table_fitted_parameters.csv when available.
COHORT_FOR_DEFAULTS = "selectedDateRaces"
VMA_KMH = 18.0
FALLBACK_ALPHA = 0.45
FALLBACK_FATIGUE_COEF = 0.40
FALLBACK_FATIGUE_MODEL = "linear"

# HRR/fatigue assumptions.
HRR_REFERENCE = 0.70
TRIMP_SCALE = 10.0
DECAY_LAMBDA = 0.30
FATIGUE_INPUT_COL = "cumTrimpBefore"
LOAD_FACTOR = 1.0
FALLBACK_HRR = 0.70

In [ ]:
def load_stage3_defaults(cohort: str = COHORT_FOR_DEFAULTS) -> dict[str, object]:
    path = PAPER_ASSET_DIR / "table_fitted_parameters.csv"
    defaults: dict[str, object] = {
        "alpha": FALLBACK_ALPHA,
        "fatigueCoef": FALLBACK_FATIGUE_COEF,
        "fatigueModel": FALLBACK_FATIGUE_MODEL,
    }
    if not path.exists():
        return defaults
    params = pd.read_csv(path)
    match = params[
        params["cohort"].astype(str).eq(cohort)
        & params["stage"].astype(str).eq("Stage 3 HRR speed ratio")
    ]
    if match.empty:
        return defaults
    row = match.iloc[0]
    defaults["alpha"] = float(row.get("alpha", defaults["alpha"]))
    defaults["fatigueCoef"] = float(row.get("fatigueCoef", defaults["fatigueCoef"]))
    defaults["fatigueModel"] = str(row.get("fatigueModel", defaults["fatigueModel"]) or defaults["fatigueModel"])
    return defaults


def load_repo_tables() -> tuple[pd.DataFrame, pd.Series]:
    activities = pd.read_csv(DATA_DIR / "activities.csv", dtype={"activityId": str})
    metrics = pd.read_csv(DATA_DIR / "activities_metrics.csv", dtype={"activityId": str})
    athlete = pd.read_csv(DATA_DIR / "athlete.csv").iloc[0]
    metric_cols = [
        col
        for col in ["activityId", "category", "timeSec", "distanceEqKm", "trimp", "startDate"]
        if col in metrics.columns
    ]
    merged = activities.merge(metrics[metric_cols], on="activityId", how="left", suffixes=("", "_metric"))
    return merged, athlete


activity_df, athlete = load_repo_tables()
HR_REST = float(athlete.get("hrRest", 55.0))
HR_MAX = float(athlete.get("hrMax", 205.0))
stage3_defaults = load_stage3_defaults()
stage3_defaults

In [ ]:
def activity_candidates(limit: int = 80) -> pd.DataFrame:
    candidates = activity_df.copy()
    candidates["activityId"] = candidates["activityId"].astype(str)
    candidates["timeseriesPath"] = candidates["activityId"].map(lambda value: TIMESERIES_DIR / f"{value}.csv")
    candidates = candidates[candidates["timeseriesPath"].map(lambda path: path.exists())].copy()
    category = candidates.get("category", pd.Series("", index=candidates.index)).astype(str).str.upper()
    sport_type = candidates.get("sportType", pd.Series("", index=candidates.index)).astype(str)
    candidates = candidates[category.isin({"RUN", "TRAIL_RUN"}) | sport_type.str.contains("run", case=False, na=False)].copy()
    candidates["distanceKm"] = pd.to_numeric(candidates.get("distanceKm"), errors="coerce")
    candidates["ascentM"] = pd.to_numeric(candidates.get("ascentM"), errors="coerce")
    candidates["movingSec"] = pd.to_numeric(candidates.get("movingSec"), errors="coerce")
    candidates["avgHr"] = pd.to_numeric(candidates.get("avgHr"), errors="coerce")
    candidates["hrReserveRatio"] = ((candidates["avgHr"] - HR_REST) / (HR_MAX - HR_REST)).clip(0.0, 1.2)
    candidates["score"] = candidates["distanceKm"].fillna(0.0) + 0.01 * candidates["ascentM"].fillna(0.0)
    columns = [
        "activityId",
        "startTime",
        "name",
        "sportType",
        "category",
        "distanceKm",
        "ascentM",
        "movingSec",
        "avgHr",
        "hrReserveRatio",
        "score",
    ]
    return candidates.sort_values("score", ascending=False)[columns].head(limit).reset_index(drop=True)


candidates = activity_candidates()
display(candidates.head(20))

if ACTIVITY_ID not in set(candidates["activityId"].astype(str)) and not (TIMESERIES_DIR / f"{ACTIVITY_ID}.csv").exists():
    ACTIVITY_ID = str(candidates.iloc[0]["activityId"])
ACTIVITY_ID

In [ ]:
def load_activity_timeseries(activity_id: str) -> pd.DataFrame:
    path = TIMESERIES_DIR / f"{activity_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No timeseries file found for {activity_id}: {path}")
    raw = pd.read_csv(path)
    prepared = tpm.prepare_raw_timeseries_for_segments(raw)
    if prepared.empty:
        raise ValueError(f"Activity {activity_id} has no usable prepared timeseries")
    return prepared


def build_activity_prediction(activity_id: str) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    prepared = load_activity_timeseries(activity_id)
    segments = tpm.segment_timeseries(
        prepared,
        segment_km=SEGMENT_KM,
        hr_rest=HR_REST,
        hr_max=HR_MAX,
    )
    if segments.empty:
        raise ValueError(f"Activity {activity_id} produced no usable segments")
    segments = segments.copy()
    segments["activityId"] = str(activity_id)
    if "terrainFamily" not in segments.columns:
        segments["terrainFamily"] = segments["avgGrade"].map(tpm.terrain_family)
    else:
        segments["terrainFamily"] = segments["terrainFamily"].fillna(segments["avgGrade"].map(tpm.terrain_family))

    meta_rows = activity_df[activity_df["activityId"].astype(str).eq(str(activity_id))]
    meta = meta_rows.iloc[0] if not meta_rows.empty else pd.Series(dtype=object)
    fallback_hrr = float(meta.get("avgHr", np.nan) - HR_REST) / (HR_MAX - HR_REST) if pd.notna(meta.get("avgHr", np.nan)) else FALLBACK_HRR
    fallback_hrr = float(np.clip(fallback_hrr, 0.0, 1.2))

    prediction = tpm.simulate_observed_hrr_segments(
        segments,
        v_anchor_kmh=VMA_KMH,
        alpha=float(stage3_defaults["alpha"]),
        fatigue_coef=float(stage3_defaults["fatigueCoef"]),
        fatigue_model=str(stage3_defaults["fatigueModel"]),
        hrr_reference=HRR_REFERENCE,
        trimp_scale=TRIMP_SCALE,
        decay_lambda=DECAY_LAMBDA,
        fatigue_input_col=FATIGUE_INPUT_COL,
        load_factor=LOAD_FACTOR,
        fallback_hrr=fallback_hrr,
    )
    prediction["actualMin"] = prediction["actualTimeSec"] / 60.0
    prediction["predictedMin"] = prediction["predictedTimeSec"] / 60.0
    prediction["errorMin"] = prediction["errorSec"] / 60.0
    prediction["cumulativeActualMin"] = prediction["cumulativeActualTimeSec"] / 60.0
    prediction["cumulativePredictedMin"] = prediction["cumulativePredictedTimeSec"] / 60.0
    prediction["cumulativeErrorMin"] = prediction["cumulativeErrorSec"] / 60.0
    prediction["gradePct"] = 100.0 * prediction["avgGrade"]
    prediction["gapCostPct"] = 100.0 * (prediction.get("gapFactorIntegrated", 1.0) - 1.0)
    prediction["climbSharePct"] = 100.0 * prediction.get("climbShare", 0.0)
    prediction["descentSharePct"] = 100.0 * prediction.get("descentShare", 0.0)
    prediction["segmentLabel"] = prediction["segmentIndex"].astype(int).astype(str)
    return prediction, meta, prepared


def activity_summary(prediction: pd.DataFrame, meta: pd.Series) -> pd.DataFrame:
    actual_sec = float(prediction["actualTimeSec"].sum())
    predicted_sec = float(prediction["predictedTimeSec"].sum())
    distance_km = float(prediction["distanceKm"].sum())
    ascent_m = float(prediction["elevGainM"].sum())
    mean_hrr = float(np.average(prediction["meanHrReserve"].fillna(FALLBACK_HRR), weights=prediction["actualTimeSec"].clip(lower=1.0)))
    return pd.DataFrame(
        [
            {
                "activityId": meta.get("activityId", ""),
                "name": meta.get("name", ""),
                "distanceKm": distance_km,
                "ascentM": ascent_m,
                "meanObservedHRR": mean_hrr,
                "actualMin": actual_sec / 60.0,
                "predictedMin": predicted_sec / 60.0,
                "errorMin": (predicted_sec - actual_sec) / 60.0,
                "segmentMAEmin": float(prediction["errorMin"].abs().mean()),
                "stageAlpha": float(stage3_defaults["alpha"]),
                "fatigueCoef": float(stage3_defaults["fatigueCoef"]),
                "fatigueModel": str(stage3_defaults["fatigueModel"]),
            }
        ]
    )

In [ ]:
SEGMENT_FILL_COLORS = {
    "steep_climb": "rgba(214, 39, 40, 0.20)",
    "climb": "rgba(245, 133, 24, 0.18)",
    "flat": "rgba(128, 128, 128, 0.10)",
    "descent": "rgba(76, 120, 168, 0.16)",
    "steep_descent": "rgba(84, 39, 143, 0.18)",
    "mixed_climb_descent": "rgba(44, 160, 44, 0.16)",
}
SEGMENT_LINE_COLORS = {
    "steep_climb": "#d62728",
    "climb": "#f58518",
    "flat": "#777777",
    "descent": "#4c78a8",
    "steep_descent": "#54278f",
    "mixed_climb_descent": "#2ca02c",
}


def elevation_segmentation_figure(prepared: pd.DataFrame, prediction: pd.DataFrame, title: str) -> go.Figure:
    elevation_col = "elevationM_ma_5" if "elevationM_ma_5" in prepared.columns else "elevationM"
    fig = go.Figure()
    for row in prediction.itertuples(index=False):
        terrain = str(getattr(row, "terrainFamily", "flat"))
        start_km = float(getattr(row, "startKm"))
        end_km = float(getattr(row, "endKm"))
        fig.add_vrect(
            x0=start_km,
            x1=end_km,
            fillcolor=SEGMENT_FILL_COLORS.get(terrain, SEGMENT_FILL_COLORS["flat"]),
            line_width=0,
            layer="below",
        )
        fig.add_vline(
            x=end_km,
            line_width=1,
            line_dash="dot",
            line_color=SEGMENT_LINE_COLORS.get(terrain, SEGMENT_LINE_COLORS["flat"]),
        )
    fig.add_trace(
        go.Scatter(
            x=prepared["cumulated_distance"],
            y=prepared[elevation_col],
            mode="lines",
            name="Full elevation profile",
            line=dict(color="#111111", width=2),
            hovertemplate="Distance %{x:.2f} km<br>Elevation %{y:.0f} m<extra></extra>",
        )
    )
    segment_mid = (prediction["startKm"] + prediction["endKm"]) / 2.0
    fig.add_trace(
        go.Scatter(
            x=segment_mid,
            y=prediction["meanAltitudeM"],
            mode="markers+text",
            text=prediction["segmentIndex"].astype(int).astype(str),
            textposition="top center",
            name="Segment midpoint",
            marker=dict(
                size=8,
                color=prediction["gradePct"],
                colorscale="RdBu_r",
                cmin=-20,
                cmax=20,
                colorbar=dict(title="Grade %"),
            ),
            customdata=np.stack(
                [
                    prediction["startKm"],
                    prediction["endKm"],
                    prediction["gradePct"],
                    prediction["gapCostPct"],
                    prediction["climbSharePct"],
                    prediction["descentSharePct"],
                    prediction["terrainFamily"],
                ],
                axis=-1,
            ),
            hovertemplate="Segment %{text}<br>%{customdata[0]:.1f}-%{customdata[1]:.1f} km<br>Mean altitude %{y:.0f} m<br>Grade %{customdata[2]:.1f}%<br>Integrated GAP cost %{customdata[3]:.1f}%<br>Climb/descent %{customdata[4]:.0f}%/%{customdata[5]:.0f}%<br>%{customdata[6]}<extra></extra>",
        )
    )
    for terrain, color in SEGMENT_LINE_COLORS.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(size=10, color=color),
                name=terrain,
            )
        )
    fig.update_layout(
        title=f"Full elevation profile and segment cuts | {title}",
        height=620,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
        xaxis_title="Distance (km)",
        yaxis_title="Elevation (m)",
    )
    return fig


def prediction_figure(prediction: pd.DataFrame, title: str) -> go.Figure:
    x = prediction["endKm"]
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        specs=[[{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}]],
        subplot_titles=(
            "Elevation and grade",
            "Segment time: actual vs predicted",
            "Observed HRR and acute performance reserve",
            "Cumulative actual vs predicted time",
        ),
    )
    fig.add_trace(go.Scatter(x=x, y=prediction["meanAltitudeM"], name="Elevation m", mode="lines", line=dict(color="#4c78a8")), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Bar(x=x, y=prediction["gradePct"], name="Grade %", marker_color="#f58518", opacity=0.45), row=1, col=1, secondary_y=True)

    fig.add_trace(go.Bar(x=x, y=prediction["actualMin"], name="Actual segment min", marker_color="#72b7b2", opacity=0.70), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Bar(x=x, y=prediction["predictedMin"], name="Predicted segment min", marker_color="#e45756", opacity=0.55), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=prediction["errorMin"], name="Prediction error min", mode="lines+markers", line=dict(color="#2f4b7c")), row=2, col=1, secondary_y=True)

    fig.add_trace(go.Scatter(x=x, y=prediction["meanHrReserve"], name="Observed segment HRR", mode="lines+markers", line=dict(color="#54a24b")), row=3, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=prediction["predictedFatigueStateBefore"], name="Predicted acute performance reserve", mode="lines", line=dict(color="#b279a2")), row=3, col=1, secondary_y=True)

    fig.add_trace(go.Scatter(x=x, y=prediction["cumulativeActualMin"], name="Cumulative actual min", mode="lines", line=dict(color="#72b7b2")), row=4, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=prediction["cumulativePredictedMin"], name="Cumulative predicted min", mode="lines", line=dict(color="#e45756")), row=4, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=prediction["cumulativeErrorMin"], name="Cumulative error min", mode="lines", line=dict(color="#2f4b7c", dash="dot")), row=4, col=1, secondary_y=True)

    fig.update_layout(title=title, height=1100, barmode="overlay", hovermode="x unified", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0))
    fig.update_xaxes(title_text="Distance (km)", row=4, col=1)
    fig.update_yaxes(title_text="Elevation (m)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Grade (%)", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Segment time (min)", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Error (min)", row=2, col=1, secondary_y=True)
    fig.update_yaxes(title_text="HRR", row=3, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Reserve multiplier", range=[0, 1.05], row=3, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Time (min)", row=4, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Error (min)", row=4, col=1, secondary_y=True)
    return fig


def actual_vs_predicted_figure(prediction: pd.DataFrame) -> go.Figure:
    limit = float(max(prediction["actualMin"].max(), prediction["predictedMin"].max()) * 1.10)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=prediction["actualMin"],
            y=prediction["predictedMin"],
            mode="markers+text",
            text=prediction["segmentLabel"],
            textposition="top center",
            marker=dict(size=9, color=prediction["meanHrReserve"], colorscale="Viridis", colorbar=dict(title="HRR")),
            name="Segments",
            customdata=np.stack([prediction["endKm"], prediction["gradePct"], prediction["terrainFamily"]], axis=-1),
            hovertemplate="Actual %{x:.1f} min<br>Predicted %{y:.1f} min<br>End km %{customdata[0]:.1f}<br>Grade %{customdata[1]:.1f}%<br>%{customdata[2]}<extra></extra>",
        )
    )
    fig.add_trace(go.Scatter(x=[0, limit], y=[0, limit], mode="lines", line=dict(color="black"), name="identity"))
    fig.add_trace(go.Scatter(x=[0, limit], y=[0, limit * 1.10], mode="lines", line=dict(color="gray", dash="dot"), name="+10%"))
    fig.add_trace(go.Scatter(x=[0, limit], y=[0, limit * 0.90], mode="lines", line=dict(color="gray", dash="dot"), name="-10%"))
    fig.update_layout(title="Segment predicted vs actual time", height=600, xaxis_title="Actual segment time (min)", yaxis_title="Predicted segment time (min)")
    fig.update_xaxes(range=[0, limit])
    fig.update_yaxes(range=[0, limit])
    return fig

In [ ]:
def render_activity(activity_id: str) -> pd.DataFrame:
    prediction, meta, prepared = build_activity_prediction(str(activity_id))
    title = f"{meta.get('name', activity_id)} | {activity_id}"
    summary = activity_summary(prediction, meta)
    display(summary.round(3))
    display(
        prediction[
            [
                "segmentIndex",
                "startKm",
                "endKm",
                "distanceKm",
                "elevGainM",
                "elevLossM",
                "gradePct",
                "gapCostPct",
                "climbSharePct",
                "descentSharePct",
                "gradeSwitchCount",
                "terrainFamily",
                "meanHrReserve",
                "predictedFatigueStateBefore",
                "actualMin",
                "predictedMin",
                "errorMin",
                "cumulativeErrorMin",
            ]
        ].round(3)
    )
    elevation_segmentation_figure(prepared, prediction, title).show()
    prediction_figure(prediction, title).show()
    actual_vs_predicted_figure(prediction).show()
    return prediction


prediction = render_activity(ACTIVITY_ID)

## Optional Interactive Selector

If `ipywidgets` is installed in the active kernel, use the dropdown to rerender another activity without editing the config cell.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets is not installed in this environment. Edit ACTIVITY_ID in the config cell and rerun render_activity(...).")
else:
    options = []
    for _, row in candidates.iterrows():
        label = f"{row['startTime']} | {row['activityId']} | {str(row['name'])[:55]} | {row['distanceKm']:.1f} km | D+ {row['ascentM']:.0f} m"
        options.append((label, row["activityId"]))
    dropdown = widgets.Dropdown(options=options, value=ACTIVITY_ID if ACTIVITY_ID in [value for _, value in options] else options[0][1], description="Activity", layout=widgets.Layout(width="95%"))
    output = widgets.Output()

    def _rerender(change: object | None = None) -> None:
        with output:
            output.clear_output(wait=True)
            render_activity(str(dropdown.value))

    dropdown.observe(_rerender, names="value")
    display(widgets.VBox([dropdown, output]))
    _rerender()